[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Reproducibilidad y arranque del Proyecto integrador.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete · Módulo IV: *Proyecto integrador*

> 📝 **Versión del alumno.** Las celdas marcadas con `# TU CÓDIGO AQUÍ` y las de *Tu respuesta* son para ti.

En este *notebook* se revisarán las buenas prácticas de reproducibilidad y se arrancará el Proyecto integrador con la plantilla `proyecto/plantilla_proyecto_integrador.ipynb`: descarga del *dataset*, preguntas de análisis y auditoría inicial.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* 🟢 **Ejercicio básico:** lo que todos deben dominar.
* 🔴 **Reto (opcional):** para quien termina antes o quiere profundizar. Si no da tiempo en clase, se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.

## 📚 Referencias.

Este *notebook* integra lo visto en las sesiones 1 a 6. Para la API: [py311 · 20 Limpieza y datos faltantes](https://github.com/PythonistaMX/py311/blob/main/20_limpieza_y_datos_faltantes.ipynb), [py311 · 24 Métodos `groupby()`](https://github.com/PythonistaMX/py311/blob/main/24_metodos_groupby.ipynb), [py311 · 31 Introducción a *Seaborn*](https://github.com/PythonistaMX/py311/blob/main/31_introduccion_a_seaborn.ipynb) y la documentación de [*SciPy* · `scipy.stats`](https://docs.scipy.org/doc/scipy/reference/stats.html).

> ⚠️ **Nota:** py311 se escribió para *pandas* 2 y este curso usa *pandas* 3; si algo se ve distinto, se sigue lo que muestra este *notebook*.

### Preparación del entorno.

* La siguiente celda importará las bibliotecas y definirá `DATA` y la semilla `rng`.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.oneway import anova_oneway

### Objetos de los *notebooks* anteriores.

Este *notebook* es independiente: la siguiente celda vuelve a crear lo que se necesita de los *notebooks* anteriores.

* La siguiente celda definirá las funciones reutilizables `perfil_rapido` y `resumen_por_grupo` y recreará `autos_limpio` con las decisiones de `23_eda_completo_con_mpg.ipynb`: `origin` como categoría, `litros_100km` y cuartiles de peso.

In [ ]:
def perfil_rapido(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "n_faltantes": df.isna().sum(),
        "pct_faltantes": (df.isna().mean() * 100).round(1),
        "n_unicos": df.nunique(dropna=True),
        "ejemplo": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })


def resumen_por_grupo(df, columna, grupo):
    return df.groupby(grupo, observed=True)[columna].agg(
        n="count", media="mean", mediana="median", desv_est="std",
        iqr=lambda s: s.quantile(0.75) - s.quantile(0.25),
    ).round(2)


autos = pd.read_csv(os.path.join(DATA, "mpg.csv"))
autos_limpio = (
    autos.astype({"origin": pd.CategoricalDtype(["usa", "europe", "japan"])})
    .assign(litros_100km=lambda d: 235.215 / d["mpg"])
    .assign(cuartil_peso=lambda d: pd.qcut(d["weight"], 4, labels=["Q1 ligero", "Q2", "Q3", "Q4 pesado"]))
)
autos_limpio.shape

---
## 1. Buenas prácticas de reproducibilidad (15 min).

| Práctica | Cómo | Por qué |
|---|---|---|
| **Restart & Run All** antes de compartir | Menú *Kernel* → *Restart Kernel and Run All Cells* | Un *notebook* que solo corre en cierto orden no es reproducible |
| Orden fijo | Preparación → datos → calidad → descripción → relaciones → inferencia → informe | Quien lo lea sigue el razonamiento |
| Semilla | `rng = np.random.default_rng(401)` en la preparación y `random_state=401` en `bootstrap`/`permutation_test` | Mismos números en cada ejecución |
| Rutas relativas | `os.path.join(DATA, "archivo.csv")` | Funciona en *Codespaces*, en tu máquina y en la del revisor |
| Datos crudos intactos | Leer el original y guardar el limpio con otro nombre (`*_limpio.csv`) | Siempre se puede rehacer la limpieza |
| Funciones reutilizables | `perfil_rapido`, `resumen_por_grupo`, `d_cohen`… | Menos errores de copiar y pegar |
| Dependencias | Si usas una biblioteca nueva, agrégala a `requirements.txt` | El *Codespace* del revisor la instalará al crearse |
| Decisiones escritas | Una celda de Markdown por decisión: qué, por qué y cuántas filas afecta | Quien lea el análisis puede entender y revisar cada decisión |

**Nota:** en *Codespaces*, `requirements.txt` se instala solo al crear el entorno. Si lo modificas con el *Codespace* abierto, ejecuta `pip install -r requirements.txt` en la terminal.

**¿Por qué importa?** Un análisis se verifica ejecutándolo de nuevo en otro *Codespace*. Si no corre de principio a fin con los mismos números, nadie puede verificar los hallazgos y el resto del trabajo pierde credibilidad. Las prácticas de la tabla son baratas si se aplican desde el inicio y muy costosas si se intentan al final.

---
## 2. Taller: arranque del proyecto (35 min).

Copia la plantilla `proyecto/plantilla_proyecto_integrador.ipynb` a la raíz de tu repositorio con un nombre propio (por ejemplo, `proyecto_diamonds.ipynb`) y trabaja ahí. Los siguientes ejercicios son el primer avance.

**Nota:** los *datasets* del proyecto no vienen en el repositorio: se descargan a `data/proyecto/` (carpeta ignorada por *git*) con el script `tools/descargar_datasets.py` o con la celda de la sección 2.1. La fuente, la licencia, la cita y el tamaño de cada uno están en `data/DATASETS.md`.

### 2.1 Descargar el *dataset* del proyecto.

| Clave | *Dataset* | Filas × columnas | Tamaño del CSV | Pregunta típica |
|---|---|---|---|---|
| `diamonds` | Precios de diamantes (*Seaborn* / *ggplot2*) | 53,940 × 10 | 2.6 MiB | ¿Cómo se relaciona el precio con el peso, el corte, el color y la claridad? |
| `adult` | Adult / Census Income, censo de EE. UU. de 1994 (UCI) | 48,842 × 15 | 5.1 MiB | ¿Qué características se asocian con ingresos mayores de 50,000 dólares anuales? |
| `bike` | Bike Sharing, rentas por hora en Washington, D. C., 2011-2012 (UCI) | 17,379 × 17 | 1.1 MiB | ¿Cómo cambian las rentas con la hora, la temporada y el clima? |
| `wine` | Wine Quality, vinos *vinho verde* tintos y blancos (UCI) | 6,497 × 13 | 0.4 MiB | ¿Qué propiedades fisicoquímicas se asocian con la calidad? |

**Desde la terminal del *Codespace*** (en la raíz del repositorio):

```
python tools/descargar_datasets.py --lista
python tools/descargar_datasets.py <clave>
python tools/descargar_datasets.py <clave> --originales
```

Donde:
* `<clave>` es `diamonds`, `adult`, `bike`, `wine` o `todos`.
* `--originales` descarga además el ZIP oficial del UCI con su documentación (y, en Bike Sharing, el archivo diario `day.csv`).

**Desde el *notebook*:** escribe la clave en `DATASET_PROYECTO` y ejecuta la celda siguiente. Si el archivo ya está en `data/proyecto/`, no se vuelve a descargar. Si no hay internet o la descarga falla, la celda lo avisa y carga `mpg.csv`, así que el resto del *notebook* sigue corriendo.

**Advertencia:** en Adult, los faltantes vienen como `NaN` **y** como el texto `"?"`, y la columna `income` tiene dos escrituras de cada clase (`">50K"` y `">50K."`). No se corrigen al descargar: son parte de tu auditoría de calidad.

* La siguiente celda descargará (si hace falta) y cargará el *dataset* elegido en `DATASET_PROYECTO`, con `mpg.csv` como respaldo si no se eligió ninguno o si la descarga falla.

In [ ]:
import sys

DATASET_PROYECTO = None  # elige: "diamonds", "adult", "bike" o "wine"
ruta_proyecto = os.path.join(DATA, "mpg.csv")
if DATASET_PROYECTO:
    ruta_descargada = os.path.join(DATA, "proyecto", f"{DATASET_PROYECTO}.csv")
    try:
        if not os.path.isfile(ruta_descargada):
            sys.path.insert(0, os.path.join(os.path.dirname(DATA), "tools"))
            from descargar_datasets import descargar
            descargar(DATASET_PROYECTO)
        ruta_proyecto = ruta_descargada
    except Exception as error:
        print(f"Aviso: no se pudo obtener {DATASET_PROYECTO!r} ({error}). Se usará mpg.csv; consulta data/DATASETS.md.")
proyecto = pd.read_csv(ruta_proyecto)
os.path.basename(ruta_proyecto), proyecto.shape

**Plantilla del informe.** La plantilla del proyecto ya trae, al final, la estructura del informe de hallazgos que se trabajará en la sesión 8: **resumen ejecutivo** (máximo 150 palabras), **datos y método**, **hallazgos** (uno por pregunta, con su gráfica, su número clave, su incertidumbre y su tamaño del efecto), **limitaciones** y **reproducibilidad**. La sección 2.6 de `23_eda_completo_con_mpg.ipynb` es un ejemplo breve de ese formato.

### 🟢 Ejercicio 1 · Preguntas de análisis.

**🎯 Objetivo:** formular el primer producto del proyecto. **Qué se espera aprender:** a escribir preguntas **respondibles** con las columnas disponibles, que guíen el resto del análisis y anticipen confusores.

Escribe 2 o 3 preguntas de análisis para tu *dataset*. Cada una debe poder responderse con las columnas disponibles, nombrar las variables involucradas y anticipar al menos un confusor o sesgo posible.

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🟢 Ejercicio 2 · Auditoría inicial.

**🎯 Objetivo:** aplicar las funciones reutilizables a tu propio *dataset*. **Qué se espera aprender:** a convertir los resultados de la auditoría en decisiones escritas, con su justificación y el número de filas afectadas.

Con el `proyecto` que cargó la sección 2.1 (o con `mpg.csv`, mientras no elijas *dataset*), aplica `perfil_rapido`, el conteo de duplicados (exactos y por la llave que corresponda) y `describe()`. Escribe en Markdown las decisiones de calidad que tomarías. Si usas un *dataset* propio aprobado, colócalo en `data/proyecto/` y léelo con `pd.read_csv`.

* La siguiente celda es para tu solución.

In [ ]:
perfil_rapido(proyecto)  # TU CÓDIGO AQUÍ: agrega duplicados y describe()

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🟢 Ejercicio 3 · Consumo por número de cilindros.

**🎯 Objetivo:** repetir el flujo de comparación de grupos con otra variable. **Qué se espera aprender:** a cuestionar si una diferencia significativa y grande es un hallazgo **independiente** de otras variables correlacionadas.

En `autos_limpio`, compara `litros_100km` entre autos de 4, 6 y 8 cilindros (excluye 3 y 5): resumen por grupo, ANOVA de Welch, η² y un diagrama de caja. ¿Tiene sentido esta comparación como hallazgo independiente del peso?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🔴 Reto 1 · ¿Los autos se volvieron más eficientes con los años?

**🎯 Objetivo:** analizar una tendencia temporal con un posible confusor. **Qué se espera aprender:** a separar, de forma descriptiva, la parte de un cambio que se explica por otra variable.

Grafica la mediana del consumo y del peso por `model_year`. ¿Cuánto bajó el consumo de 1970 a 1982? ¿Qué parte de esa mejora podría deberse a que los autos se hicieron más ligeros? (pista: compara el consumo por año dentro de un mismo cuartil de peso).

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🔴 Reto 2 · ¿Importa cómo se tratan los faltantes de `horsepower`?

**🎯 Objetivo:** medir la sensibilidad de un resultado a la decisión sobre los faltantes. **Qué se espera aprender:** que el impacto de una imputación depende de cuántos valores faltan.

Compara la correlación de `horsepower` con `litros_100km` (a) eliminando los 6 faltantes, (b) imputando la mediana global y (c) imputando la mediana por número de cilindros. ¿Cambia la conclusión? ¿Por qué el resultado es distinto al de la sesión 2?

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## ✅ Cierre.

**Hoy vimos:**
* La estructura de un EDA y su aplicación completa a `mpg`, con cada decisión documentada.
* Una decisión de escala (L/100 km) que vuelve lineal la relación con el peso.
* Un confusor real: la diferencia de consumo por origen casi desaparece a igual peso.
* Buenas prácticas: **Restart & Run All**, semillas, rutas relativas, datos crudos intactos, funciones y `requirements.txt`.

**Avance del proyecto:** preguntas de análisis y auditoría de calidad en tu copia de la plantilla.

**Siguiente *notebook*:** `25_estructura_del_informe_y_comunicacion.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>